In [ ]:
import tensorflow as tf

def masked_mse(y_true, y_pred):
    mask = tf.cast(tf.not_equal(y_true, -1.0), tf.float32)
    mse  = tf.square(y_true - y_pred)
    return tf.reduce_sum(mse * mask) / tf.reduce_sum(mask)

In [ ]:
from tensorflow.keras.models import load_model
model = load_model(
    '',
    custom_objects={'masked_mse': masked_mse}
)

In [ ]:
model.summary()

In [ ]:
import os

zip_file_name = ''
unzipped_dir_name = ''

if not os.path.exists(unzipped_dir_name):
    print(f"Raspakujem {zip_file_name}...")
    !unzip -q {zip_file_name}
    print(f"'{zip_file_name}' uspešno raspakovan u '{unzipped_dir_name}'.")
else:
    print(f"Folder '{unzipped_dir_name}' već postoji. Preskačem raspakivanje.")

In [ ]:
import os

BASE_DIR = ""

def load_dataset(split):
    images_dir = os.path.join(BASE_DIR, "images", split)
    labels_dir = os.path.join(BASE_DIR, "labels", split)

    data = []

    for img_name in os.listdir(images_dir):
        if not img_name.lower().endswith((".jpg", ".png", ".jpeg")):
            continue

        base = os.path.splitext(img_name)[0]
        label_path = os.path.join(labels_dir, base + ".txt")

        if not os.path.exists(label_path):
            continue

        with open(label_path) as f:
            parts = f.read().strip().split()

        if len(parts) != 3:
            continue

        class_id = int(parts[0])
        start_x = float(parts[1])
        end_x = float(parts[2])

        data.append({
            "image_path": os.path.join(images_dir, img_name),
            "class_id": class_id,
            "start_x": start_x,
            "end_x": end_x
        })

    return data


train_data = load_dataset("train")
val_data   = load_dataset("val")
test_data = load_dataset("test")

print(f"Train samples: {len(train_data)}")
print(f"Val samples:   {len(val_data)}")
print("Test samples:", len(test_data))


In [ ]:
import numpy as np
import tensorflow as tf
from PIL import Image
import math

BATCH_SIZE = 128

class DataGenerator(tf.keras.utils.Sequence):

    def __init__(self, data, shuffle=True):
        self.data = data
        self.shuffle = shuffle
        self.indexes = np.arange(len(data))

        img = Image.open(data[0]["image_path"]).convert("RGB")
        self.H, self.W = img.size[1], img.size[0]

        self.on_epoch_end()

    def __len__(self):
        return math.ceil(len(self.data) / BATCH_SIZE)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __getitem__(self, idx):
        batch_idx = self.indexes[idx * BATCH_SIZE:(idx + 1) * BATCH_SIZE]
        batch = [self.data[i] for i in batch_idx]

        X = np.zeros((len(batch), self.H, self.W, 3), dtype=np.float32)
        y_class = np.zeros((len(batch),), dtype=np.int32)
        y_start = np.zeros((len(batch), 1), dtype=np.float32)
        y_end   = np.zeros((len(batch), 1), dtype=np.float32)

        valid = 0

        for item in batch:
            img = np.array(
                Image.open(item["image_path"]).convert("RGB"),
                dtype=np.float32
            ) / 255.0

            start = float(item["start_x"])
            end   = float(item["end_x"])


            raw_cls = int(item["class_id"])

            if raw_cls == -1:
                cls = 2
                start, end = 0.0, 0.0
            else:
                cls = raw_cls

                if not (0.0 <= start <= 1.0 and 0.0 <= end <= 1.0):
                    continue
                if start == end:
                    continue
                if start > end:
                    start, end = end, start


            X[valid] = img
            y_class[valid] = cls
            y_start[valid, 0] = start
            y_end[valid, 0]   = end
            valid += 1

        return X[:valid], {
            "class_output": y_class[:valid],
            "start_output": y_start[:valid],
            "end_output": y_end[:valid]
        }

In [ ]:
train_gen = DataGenerator(train_data, shuffle=True)
val_gen   = DataGenerator(val_data, shuffle=False)
test_gen  = DataGenerator(test_data, shuffle=False)
print(test_gen.data[1])

In [ ]:
import time
import numpy as np

times = []

for i in range(10):
    X, _ = test_gen[i]

    start = time.time()
    model.predict(X, verbose=0)
    end = time.time()

    times.append((end - start) / len(X))

print("Prosečno vreme po slici:", np.mean(times), "sekundi")

In [ ]:
from collections import Counter

all_labels = []
for d in test_data:
    all_labels.append(d["class_id"])

print("Originalne klase u test_data:", Counter(all_labels))

In [ ]:
import tensorflow as tf

def masked_mse(y_true, y_pred):
    mask = tf.cast(tf.not_equal(y_true, 0.0), tf.float32)
    return tf.reduce_sum(mask * tf.square(y_true - y_pred)) / (tf.reduce_sum(mask) + 1e-6)

In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(5e-4),
    loss={
        "class_output": "sparse_categorical_crossentropy",
        "start_output": masked_mse,
        "end_output": masked_mse
    },
    loss_weights={
        "class_output": 1.0,
        "start_output": 5.0,
        "end_output": 5.0
    },
    metrics={
        "class_output": "accuracy"
    }
)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
y_true, y_pred = [], []

for i in range(len(test_gen)):
    X, y = test_gen[i]
    preds = model.predict(X, verbose=0)
    y_pred.extend(np.argmax(preds[0], axis=1))
    y_true.extend(y["class_output"])

labels = [0, 1, 2]
names  = ["Limenka", "PET", "Neklasifikovano"]

cm = confusion_matrix(y_true, y_pred, labels=labels)

disp = ConfusionMatrixDisplay(cm, display_labels=names)
disp.plot(cmap="Blues")

plt.xlabel("Predviđena klasa")
plt.ylabel("Stvarna klasa")
plt.title("")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_true, y_pred,
    target_names=names
))